# QQQ vs Top-10 Basket Arbitrage (Demo Backtest)

This notebook is an **interview-friendly demo** of a simple ETF/basket dislocation strategy:

- **ETF:** `QQQ` (Nasdaq-100 ETF)
- **Basket:** top-10 QQQ holdings (commonly reported around early 2026): `NVDA, AAPL, MSFT, AMZN, TSLA, GOOGL, META, GOOG, AVGO, COST`  
  *(Holdings change over time; feel free to update the list.)*

What this demo focuses on:
- clean data pipeline (yfinance + caching)
- a small but correct backtest engine (cash, positions, avg cost, commissions/slippage)
- a *slightly improved* signal (rolling normalization + z-score + entry/exit bands)
- basic risk / sizing (dollar-neutral legs, gross exposure cap)

**Note:** This is not “meant” to be perfectly profitable — it’s meant to be explainable and auditable.


In [7]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import yfinance as yf
from datetime import date
from dataclasses import dataclass
from pathlib import Path

plt.rcParams["figure.figsize"] = (14, 5)


## 1) Universe + Data (yfinance, cached)

Tip for interviews: run once before the meeting to warm the cache.


In [8]:
ETF_TICKER = "QQQ"

# Top-10 holdings list (update anytime)
COMPONENTS = ["NVDA","AAPL","MSFT","AMZN","TSLA","GOOGL","META","GOOG","AVGO","COST"]

TICKERS = [ETF_TICKER] + COMPONENTS
START = "2018-01-01"
END = date.today().isoformat()

CACHE_PATH = "qqq_top10_prices.parquet"  # local cache file

def load_prices(tickers, start=START, end=END, cache_path=CACHE_PATH, force_refresh=False):
    if (not force_refresh) and Path(cache_path).exists():
        px = pd.read_parquet(cache_path)
        if all(t in px.columns for t in tickers):
            return px[tickers].copy()

    raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)

    if isinstance(raw.columns, pd.MultiIndex):
        px = raw["Close"].copy()
    else:
        px = raw.to_frame(name="Close")
        px.columns = [tickers[0]]

    px = px.dropna(how="all")
    # Drop dates with any missing ticker to keep basket aligned (simple + safe)
    px = px.dropna()

    px.to_parquet(cache_path)
    return px

prices = load_prices(TICKERS)
prices.tail()


Ticker,QQQ,NVDA,AAPL,MSFT,AMZN,TSLA,GOOGL,META,GOOG,AVGO,COST
Date,,,,,,,,,,,


## 2) Tiny Backtest Engine (cash + positions + avg cost)

- Market orders filled at close (bar close) with:
  - **commission** (bps)
  - **slippage** (bps)
- Positions tracked with **signed quantity** and **average cost**.


In [9]:
@dataclass
class Position:
    qty: float = 0.0
    avg_price: float = 0.0

class BacktestEngine:
    def __init__(self, initial_cash=100_000.0, commission_bps=1.0, slippage_bps=1.0):
        self.initial_cash = float(initial_cash)
        self.cash = float(initial_cash)
        self.commission_bps = float(commission_bps)
        self.slippage_bps = float(slippage_bps)

        self.positions = {}  # ticker -> Position
        self.realized_pnl = 0.0
        self.trades = []

    def _ensure_pos(self, ticker):
        if ticker not in self.positions:
            self.positions[ticker] = Position()
        return self.positions[ticker]

    def _costs(self, notional):
        comm = abs(notional) * (self.commission_bps / 1e4)
        slip = abs(notional) * (self.slippage_bps / 1e4)
        return comm + slip

    def execute(self, ticker, qty, price, dt):
        qty = float(qty)
        if qty == 0:
            return

        # worse fills: buys pay up, sells receive less
        fill_price = float(price) * (1.0 + (self.slippage_bps / 1e4) * np.sign(qty))
        notional = fill_price * qty
        costs = self._costs(notional)

        pos = self._ensure_pos(ticker)
        prev_qty = pos.qty

        # Realize PnL if reducing/closing against existing position
        if prev_qty != 0 and np.sign(prev_qty) != np.sign(qty):
            close_shares = min(abs(prev_qty), abs(qty))
            # If prev was long, selling closes -> pnl = (fill - avg) * close_shares
            # If prev was short, buying closes -> pnl = (avg - fill) * close_shares
            realized = (fill_price - pos.avg_price) * close_shares * np.sign(prev_qty)
            self.realized_pnl += realized

        new_qty = prev_qty + qty

        # Update avg price
        if new_qty == 0:
            pos.qty = 0.0
            pos.avg_price = 0.0
        elif prev_qty == 0 or np.sign(new_qty) != np.sign(prev_qty):
            # opening fresh or flipped
            pos.qty = new_qty
            pos.avg_price = fill_price
        else:
            # same direction: update avg only when increasing abs exposure
            if abs(new_qty) > abs(prev_qty):
                added = abs(new_qty) - abs(prev_qty)
                pos.avg_price = (pos.avg_price * abs(prev_qty) + fill_price * added) / abs(new_qty)
            pos.qty = new_qty

        # Cash: buys spend, sells raise, always pay costs
        self.cash -= notional
        self.cash -= costs

        self.trades.append({
            "date": dt, "ticker": ticker, "qty": qty, "price": fill_price,
            "notional": notional, "costs": costs
        })

    def market_value(self, price_map):
        return sum(pos.qty * float(price_map[t]) for t, pos in self.positions.items() if pos.qty != 0 and t in price_map)

    def equity(self, price_map):
        return self.cash + self.market_value(price_map)

    def gross_exposure(self, price_map):
        return sum(abs(pos.qty * float(price_map[t])) for t, pos in self.positions.items() if pos.qty != 0 and t in price_map)


## 3) Synthetic basket + spread z-score

Rolling rebase to avoid “anchoring” everything to the first day of the sample.


In [10]:
LOOKBACK_REBASE = 20
LOOKBACK_Z = 60

def build_spread_and_z(prices: pd.DataFrame, etf=ETF_TICKER, components=COMPONENTS,
                       lookback_rebase=LOOKBACK_REBASE, lookback_z=LOOKBACK_Z):
    px = prices[[etf] + components].copy()
    px0 = px.shift(lookback_rebase)

    etf_idx = (px[etf] / px0[etf]) * 100.0
    comp_rebased = (px[components].div(px0[components])) * 100.0
    syn_idx = comp_rebased.mean(axis=1)

    spread = (etf_idx - syn_idx)

    roll_mu = spread.rolling(lookback_z).mean()
    roll_sd = spread.rolling(lookback_z).std(ddof=0)

    z = (spread - roll_mu) / roll_sd
    out = pd.DataFrame({"spread": spread, "z": z}).dropna()
    return out

features = build_spread_and_z(prices)
features.tail()


,spread,z
Date,,


## 4) Strategy: bands + dollar-neutral sizing + exposure cap

- Enter at `|z| > ENTRY_Z`
- Exit at `|z| < EXIT_Z`
- Dollar-neutral legs with a modest notional per leg
- Gross exposure capped as a multiple of equity


In [11]:
ENTRY_Z = 2.0
EXIT_Z = 0.5
NOTIONAL_PER_LEG = 30_000
MAX_GROSS_MULT = 2.0

def target_basket_orders(price_row, direction, notional=NOTIONAL_PER_LEG):
    # direction: +1 long ETF/short basket, -1 short ETF/long basket
    etf_price = float(price_row[ETF_TICKER])
    n = len(COMPONENTS)

    etf_qty = direction * (notional / etf_price)
    orders = [(ETF_TICKER, etf_qty)]

    basket_sign = -direction
    for t in COMPONENTS:
        p = float(price_row[t])
        qty = basket_sign * (notional / n / p)
        orders.append((t, qty))
    return orders

def flatten_all(engine: BacktestEngine, price_row, dt):
    for t, pos in list(engine.positions.items()):
        if pos.qty != 0:
            engine.execute(t, -pos.qty, float(price_row[t]), dt)

def current_state(engine: BacktestEngine):
    pos = engine.positions.get(ETF_TICKER, Position())
    return 0 if pos.qty == 0 else int(np.sign(pos.qty))

def run_backtest(prices: pd.DataFrame, features: pd.DataFrame,
                 commission_bps=1.0, slippage_bps=1.0):
    eng = BacktestEngine(initial_cash=100_000, commission_bps=commission_bps, slippage_bps=slippage_bps)
    equity_curve, gross_curve = [], []

    for dt, row in prices.iterrows():
        price_map = row.to_dict()

        if dt in features.index:
            z = float(features.loc[dt, "z"])
            state = current_state(eng)

            # Exit
            if state != 0 and abs(z) < EXIT_Z:
                flatten_all(eng, row, dt)
                state = 0

            # Entry (only if flat)
            if state == 0:
                orders = []
                if z > ENTRY_Z:
                    orders = target_basket_orders(row, direction=-1)
                elif z < -ENTRY_Z:
                    orders = target_basket_orders(row, direction=+1)

                if orders:
                    eq = eng.equity(price_map)
                    added_gross = sum(abs(qty * float(price_map[t])) for t, qty in orders)
                    if (eng.gross_exposure(price_map) + added_gross) <= MAX_GROSS_MULT * eq:
                        for t, qty in orders:
                            eng.execute(t, qty, float(price_map[t]), dt)

        equity_curve.append((dt, eng.equity(price_map)))
        gross_curve.append((dt, eng.gross_exposure(price_map)))

    eq = pd.Series(dict(equity_curve)).sort_index()
    gross = pd.Series(dict(gross_curve)).sort_index()
    trades = pd.DataFrame(eng.trades)
    return eng, eq, gross, trades

engine, equity, gross, trades = run_backtest(prices, features, commission_bps=1.0, slippage_bps=1.0)
equity.tail(), trades.head()


(Series([], dtype: object),
 Empty DataFrame
 Columns: []
 Index: [])

## 5) Results + basic stats (believable + auditable)

If the curve looks *too good*, increase costs or tighten the bands — you want it to look realistic.


In [12]:
def drawdown(equity: pd.Series):
    peak = equity.cummax()
    return equity / peak - 1.0

eq_ret = equity.pct_change().dropna()
dd = drawdown(equity)

stats = pd.Series({
    "Start": str(equity.index.min().date()),
    "End": str(equity.index.max().date()),
    "Final Equity": float(equity.iloc[-1]),
    "Total Return": float(equity.iloc[-1] / equity.iloc[0] - 1),
    "Ann Vol": float(eq_ret.std() * np.sqrt(252)),
    "Ann Sharpe (rf=0)": float((eq_ret.mean() / (eq_ret.std()+1e-12)) * np.sqrt(252)),
    "Max Drawdown": float(dd.min()),
    "Trades": int(len(trades)),
})
stats


AttributeError: 'float' object has no attribute 'date'

In [ ]:
plt.figure()
plt.plot(equity.index, equity.values)
plt.title("Equity Curve")

plt.figure()
plt.plot(dd.index, dd.values)
plt.title("Drawdown")

plt.figure()
plt.plot(gross.index, gross.values)
plt.title("Gross Exposure ($)")

plt.show()


## 6) Quick cost sensitivity (great interview talking point)


In [ ]:
engine_hi_cost, equity_hi_cost, gross_hi_cost, trades_hi_cost = run_backtest(
    prices, features, commission_bps=3.0, slippage_bps=3.0
)

plt.figure()
plt.plot(equity.index, equity.values, label="1bp+1bp")
plt.plot(equity_hi_cost.index, equity_hi_cost.values, label="3bp+3bp")
plt.title("Equity Sensitivity to Costs")
plt.legend()
plt.show()

print("Trades @ 1bp+1bp:", len(trades), "| Trades @ 3bp+3bp:", len(trades_hi_cost))
